In [0]:
# NYC Taxi Pipeline — Shared Functions

#Funciones reutilizables del pipeline incremental:

#Landing → Bronze → Silver → Gold

#Incluye:
#- Schema Drift / Schema Evolution
#- Idempotencia y Reprocess
#- Data Quality
#- Completeness
#- Observabilidad y trazabilidad

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

import uuid
import json

In [0]:
# ============================================================
# ENVIRONMENT CONFIGURATION
# ============================================================

VALID_ENVIRONMENTS = [
    "DEV",
    "STAGING",
    "PROD"
]


CATALOG_BY_ENVIRONMENT = {
    "DEV": "nyc_taxi_dev",
    "STAGING": "nyc_taxi_staging",
    "PROD": "nyc_taxi_prod"
}


def configure_environment(environment):

    global ENVIRONMENT
    global CATALOG

    global BRONZE_TRIPS_TABLE

    global SILVER_TRIPS_TABLE
    global QUARANTINE_TABLE

    global GOLD_DAILY_METRICS_TABLE

    global BATCH_CONTROL_TABLE
    global PIPELINE_RUNS_TABLE
    global DQ_METRICS_TABLE
    global COMPLETENESS_METRICS_TABLE
    global SCHEMA_CONTRACT_TABLE
    global SCHEMA_DRIFT_EVENTS_TABLE
    global PIPELINE_TASK_RUNS_TABLE

    # --------------------------------------------------------
    # 1. Normalizar environment
    # --------------------------------------------------------

    environment = (
        str(environment)
        .strip()
        .upper()
    )

    # --------------------------------------------------------
    # 2. Validar environment
    # --------------------------------------------------------

    if environment not in VALID_ENVIRONMENTS:

        raise ValueError(
            f"INVALID_ENVIRONMENT | "
            f"environment={environment} | "
            f"allowed={VALID_ENVIRONMENTS}"
        )

    # --------------------------------------------------------
    # 3. Resolver catálogo
    # --------------------------------------------------------

    ENVIRONMENT = environment

    CATALOG = (
        CATALOG_BY_ENVIRONMENT[
            ENVIRONMENT
        ]
    )

    # --------------------------------------------------------
    # 4. Bronze
    # --------------------------------------------------------

    BRONZE_TRIPS_TABLE = (
        f"{CATALOG}."
        f"bronze."
        f"yellow_taxi_trips"
    )

    # --------------------------------------------------------
    # 5. Silver
    # --------------------------------------------------------

    SILVER_TRIPS_TABLE = (
        f"{CATALOG}."
        f"silver."
        f"yellow_taxi_trips"
    )

    QUARANTINE_TABLE = (
        f"{CATALOG}."
        f"silver."
        f"yellow_taxi_quarantine"
    )

    # --------------------------------------------------------
    # 6. Gold
    # --------------------------------------------------------

    GOLD_DAILY_METRICS_TABLE = (
        f"{CATALOG}."
        f"gold."
        f"daily_trip_metrics"
    )

    # --------------------------------------------------------
    # 7. Monitoring
    # --------------------------------------------------------

    BATCH_CONTROL_TABLE = (
        f"{CATALOG}."
        f"monitoring."
        f"batch_control"
    )

    PIPELINE_RUNS_TABLE = (
        f"{CATALOG}."
        f"monitoring."
        f"pipeline_runs"
    )

    DQ_METRICS_TABLE = (
        f"{CATALOG}."
        f"monitoring."
        f"dq_metrics"
    )

    COMPLETENESS_METRICS_TABLE = (
        f"{CATALOG}."
        f"monitoring."
        f"completeness_metrics"
    )

    SCHEMA_CONTRACT_TABLE = (
        f"{CATALOG}."
        f"monitoring."
        f"schema_contract"
    )

    SCHEMA_DRIFT_EVENTS_TABLE = (
        f"{CATALOG}."
        f"monitoring."
        f"schema_drift_events"
    )

    PIPELINE_TASK_RUNS_TABLE = (
        f"{CATALOG}."
        f"monitoring."
        f"pipeline_task_runs"
    )

    # --------------------------------------------------------
    # 8. Mostrar configuración
    # --------------------------------------------------------

    print(
        f"Environment configured | "
        f"environment={ENVIRONMENT} | "
        f"catalog={CATALOG}"
    )

In [0]:
def require_environment():

    # ---------------------------------------------------------
    # 1. Variables que deben existir
    # ---------------------------------------------------------

    required_variables = [
        "ENVIRONMENT",
        "CATALOG",
        "BRONZE_TRIPS_TABLE",
        "SILVER_TRIPS_TABLE",
        "QUARANTINE_TABLE",
        "GOLD_DAILY_METRICS_TABLE",
        "BATCH_CONTROL_TABLE",
        "PIPELINE_RUNS_TABLE",
        "DQ_METRICS_TABLE",
        "COMPLETENESS_METRICS_TABLE",
        "SCHEMA_CONTRACT_TABLE",
        "SCHEMA_DRIFT_EVENTS_TABLE",
        "PIPELINE_TASK_RUNS_TABLE"
    ]

    missing_variables = [
        variable_name
        for variable_name in required_variables
        if variable_name not in globals()
    ]

    if missing_variables:

        raise RuntimeError(
            f"ENVIRONMENT_NOT_CONFIGURED | "
            f"missing_variables={missing_variables} | "
            f"Call configure_environment(environment) "
            f"before executing pipeline functions."
        )

    # ---------------------------------------------------------
    # 2. Validar environment
    # ---------------------------------------------------------

    if ENVIRONMENT not in VALID_ENVIRONMENTS:

        raise RuntimeError(
            f"INVALID_ACTIVE_ENVIRONMENT | "
            f"environment={ENVIRONMENT}"
        )

    # ---------------------------------------------------------
    # 3. Validar catálogo
    # ---------------------------------------------------------

    expected_catalog = (
        CATALOG_BY_ENVIRONMENT[
            ENVIRONMENT
        ]
    )

    if CATALOG != expected_catalog:

        raise RuntimeError(
            f"ENVIRONMENT_CATALOG_MISMATCH | "
            f"environment={ENVIRONMENT} | "
            f"catalog={CATALOG} | "
            f"expected_catalog={expected_catalog}"
        )

    # ---------------------------------------------------------
    # 4. Construir nombres esperados de tablas
    # ---------------------------------------------------------

    expected_tables = {

        "BRONZE_TRIPS_TABLE":
            f"{expected_catalog}."
            f"bronze.yellow_taxi_trips",

        "SILVER_TRIPS_TABLE":
            f"{expected_catalog}."
            f"silver.yellow_taxi_trips",

        "QUARANTINE_TABLE":
            f"{expected_catalog}."
            f"silver.yellow_taxi_quarantine",

        "GOLD_DAILY_METRICS_TABLE":
            f"{expected_catalog}."
            f"gold.daily_trip_metrics",

        "BATCH_CONTROL_TABLE":
            f"{expected_catalog}."
            f"monitoring.batch_control",

        "PIPELINE_RUNS_TABLE":
            f"{expected_catalog}."
            f"monitoring.pipeline_runs",

        "DQ_METRICS_TABLE":
            f"{expected_catalog}."
            f"monitoring.dq_metrics",

        "COMPLETENESS_METRICS_TABLE":
            f"{expected_catalog}."
            f"monitoring.completeness_metrics",

        "SCHEMA_CONTRACT_TABLE":
            f"{expected_catalog}."
            f"monitoring.schema_contract",

        "SCHEMA_DRIFT_EVENTS_TABLE":
            f"{expected_catalog}."
            f"monitoring.schema_drift_events",

        "PIPELINE_TASK_RUNS_TABLE":
            f"{expected_catalog}."
            f"monitoring.pipeline_task_runs"
    }

    # ---------------------------------------------------------
    # 5. Validar cada tabla contra el environment activo
    # ---------------------------------------------------------

    table_mismatches = {}

    for (
        variable_name,
        expected_table
    ) in expected_tables.items():

        actual_table = globals()[
            variable_name
        ]

        if actual_table != expected_table:

            table_mismatches[
                variable_name
            ] = {
                "actual":
                    actual_table,

                "expected":
                    expected_table
            }

    if table_mismatches:

        raise RuntimeError(
            f"ENVIRONMENT_TABLE_MISMATCH | "
            f"environment={ENVIRONMENT} | "
            f"catalog={CATALOG} | "
            f"mismatches={table_mismatches}"
        )

    return True

In [0]:
def initialize_environment():

    require_environment()

    print(
        f"INITIALIZING ENVIRONMENT | "
        f"environment={ENVIRONMENT} | "
        f"catalog={CATALOG}"
    )

    # ========================================================
    # 1. TABLAS OPERACIONALES
    #    Copiamos únicamente la estructura, NO los datos.
    # ========================================================

    table_templates = {

        # Bronze
        BRONZE_TRIPS_TABLE:
            "workspace.bronze.yellow_taxi_trips",

        # Silver
        SILVER_TRIPS_TABLE:
            "workspace.silver.yellow_taxi_trips",

        QUARANTINE_TABLE:
            "workspace.silver.yellow_taxi_quarantine",

        # Gold
        GOLD_DAILY_METRICS_TABLE:
            "workspace.gold.daily_trip_metrics",

        # Monitoring
        BATCH_CONTROL_TABLE:
            "workspace.monitoring.batch_control",

        PIPELINE_RUNS_TABLE:
            "workspace.monitoring.pipeline_runs",

        DQ_METRICS_TABLE:
            "workspace.monitoring.dq_metrics",

        COMPLETENESS_METRICS_TABLE:
            "workspace.monitoring.completeness_metrics",

        SCHEMA_DRIFT_EVENTS_TABLE:
            "workspace.monitoring.schema_drift_events",

        PIPELINE_TASK_RUNS_TABLE:
            "workspace.monitoring.pipeline_task_runs"
    }

    for target_table, source_table in table_templates.items():

        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS {target_table}
            LIKE {source_table}
        """)

        print(
            f"TABLE READY | "
            f"target={target_table} | "
            f"template={source_table}"
        )

    # ========================================================
    # 2. SCHEMA CONTRACT
    #
    #    Esta tabla sí necesita datos iniciales porque contiene
    #    el contrato que utiliza el Schema Gate.
    # ========================================================

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {SCHEMA_CONTRACT_TABLE}
        LIKE workspace.monitoring.schema_contract
    """)

    schema_contract_count = (
        spark.table(
            SCHEMA_CONTRACT_TABLE
        )
        .limit(1)
        .count()
    )

    if schema_contract_count == 0:

        spark.sql(f"""
            INSERT INTO {SCHEMA_CONTRACT_TABLE}

            SELECT *
            FROM workspace.monitoring.schema_contract
        """)

        print(
            f"SCHEMA CONTRACT SEEDED | "
            f"target={SCHEMA_CONTRACT_TABLE}"
        )

    else:

        print(
            f"SCHEMA CONTRACT ALREADY INITIALIZED | "
            f"target={SCHEMA_CONTRACT_TABLE}"
        )

    # ========================================================
    # 3. FIN
    # ========================================================

    print(
        f"ENVIRONMENT INITIALIZED | "
        f"environment={ENVIRONMENT} | "
        f"catalog={CATALOG}"
    )

In [0]:
# =========================================================
# OBSERVABILITY
# =========================================================

def start_task_run(
    batch_id,
    stage
):

    require_environment()

    task_run_id = str(
        uuid.uuid4()
    )

    stage = (
        str(stage)
        .strip()
        .upper()
    )

    print(
        f"TASK START | "
        f"environment={ENVIRONMENT} | "
        f"batch={batch_id} | "
        f"stage={stage} | "
        f"task_run_id={task_run_id}"
    )

    spark.sql(f"""
        INSERT INTO {PIPELINE_TASK_RUNS_TABLE}
        (
            task_run_id,
            batch_id,
            stage,
            status,
            started_at,
            finished_at,
            error_message
        )

        VALUES (
            '{task_run_id}',
            '{batch_id}',
            '{stage}',
            'RUNNING',
            current_timestamp(),
            NULL,
            NULL
        )
    """)

    return task_run_id
    

def finish_task_run(
    task_run_id,
    status,
    error_message=None
):

    require_environment()

    status = (
        str(status)
        .strip()
        .upper()
    )

    if status not in [
        "SUCCESS",
        "FAILED"
    ]:

        raise ValueError(
            f"INVALID_TASK_STATUS | "
            f"status={status}"
        )

    if error_message is None:

        error_sql = "NULL"

    else:

        safe_error_message = (
            str(error_message)
            .replace("'", "''")
        )

        error_sql = (
            f"'{safe_error_message}'"
        )

    spark.sql(f"""
        UPDATE {PIPELINE_TASK_RUNS_TABLE}

        SET
            status = '{status}',
            finished_at = current_timestamp(),
            error_message = {error_sql}

        WHERE
            task_run_id = '{task_run_id}'
    """)

    print(
        f"TASK FINISH | "
        f"environment={ENVIRONMENT} | "
        f"task_run_id={task_run_id} | "
        f"status={status}"
    )

In [0]:
# =========================================================
# LANDING
# =========================================================

def get_landing_status(
    delivery_date,
    delivery_batch
):

    landing_path = (
        "gs://de-zoomcamp-501217-databricks/"
        "landing/nyc_taxi/yellow/"
        f"{delivery_date}/batch_{delivery_batch}/"
    )

    try:
        files = dbutils.fs.ls(landing_path)

    except Exception:

        return {
            "status": "MISSING",
            "records": None,
            "path": landing_path
        }

    parquet_files = [
        file.path
        for file in files
        if file.path.endswith(".parquet")
    ]

    if len(parquet_files) == 0:

        return {
            "status": "MISSING",
            "records": None,
            "path": landing_path
        }

    try:

        df = spark.read.parquet(
            landing_path
        )

        records = df.count()

    except Exception:

        return {
            "status": "CORRUPT",
            "records": None,
            "path": landing_path
        }

    if records == 0:

        return {
            "status": "EMPTY",
            "records": 0,
            "path": landing_path
        }

    return {
        "status": "AVAILABLE",
        "records": records,
        "path": landing_path
    }

In [0]:
# =========================================================
# SCHEMA MANAGEMENT
# =========================================================

def load_schema_contract():

    require_environment()

    contract_df = spark.table(
        SCHEMA_CONTRACT_TABLE
    )

    contract_rows = contract_df.collect()

    required_columns = {
        row["column_name"]: row["data_type"]
        for row in contract_rows
        if row["is_required"]
    }

    optional_columns = {
        row["column_name"]: row["data_type"]
        for row in contract_rows
        if not row["is_required"]
    }

    return required_columns, optional_columns

In [0]:
def detect_schema_drift_v2(incoming_df):

    require_environment()

    required_columns, optional_columns = (
        load_schema_contract()
    )

    incoming_schema = {
        field.name: field.dataType.simpleString()
        for field in incoming_df.schema.fields
    }

    incoming_names = set(
        incoming_schema.keys()
    )

    required_names = set(
        required_columns.keys()
    )

    optional_names = set(
        optional_columns.keys()
    )

    known_names = (
        required_names
        |
        optional_names
    )

    missing_required_columns = sorted(
        required_names
        -
        incoming_names
    )

    missing_optional_columns = sorted(
        optional_names
        -
        incoming_names
    )

    unknown_columns = sorted(
        incoming_names
        -
        known_names
    )

    expected_schema = {
        **required_columns,
        **optional_columns
    }

    type_changes = {}

    for column_name in (
        known_names
        &
        incoming_names
    ):

        if (
            expected_schema[column_name]
            != incoming_schema[column_name]
        ):

            type_changes[column_name] = {
                "expected":
                    expected_schema[column_name],

                "incoming":
                    incoming_schema[column_name]
            }

    return {
        "unknown_columns":
            unknown_columns,

        "missing_required_columns":
            missing_required_columns,

        "missing_optional_columns":
            missing_optional_columns,

        "type_changes":
            type_changes
    }

In [0]:
def classify_schema_change_v2(schema_check):

    unknown_columns = schema_check["unknown_columns"]
    missing_required_columns = schema_check["missing_required_columns"]
    missing_optional_columns = schema_check["missing_optional_columns"]
    type_changes = schema_check["type_changes"]

    if missing_required_columns:
        return {
            "decision": "REJECT",
            "reason": "Required columns are missing"
        }

    if type_changes:
        return {
            "decision": "REJECT",
            "reason": "Datatype changes detected"
        }

    if unknown_columns:
        return {
            "decision": "AUTO_EVOLVE",
            "reason": "New additive columns detected"
        }

    return {
        "decision": "ACCEPT",
        "reason": (
            "Schema compatible; "
            "missing optional columns are allowed"
        )
    }

In [0]:
def write_schema_drift_event_v2(
    batch_id,
    schema_check,
    schema_decision
):

    require_environment()

    event_data = [{
        "event_id":
            str(uuid.uuid4()),

        "batch_id":
            batch_id,

        "decision":
            schema_decision["decision"],

        "unknown_columns":
            json.dumps(
                schema_check[
                    "unknown_columns"
                ]
            ),

        "missing_required_columns":
            json.dumps(
                schema_check[
                    "missing_required_columns"
                ]
            ),

        "missing_optional_columns":
            json.dumps(
                schema_check[
                    "missing_optional_columns"
                ]
            ),

        "type_changes":
            json.dumps(
                schema_check[
                    "type_changes"
                ]
            )
    }]

    event_df = (
        spark.createDataFrame(
            event_data
        )
        .withColumn(
            "detected_at",
            F.current_timestamp()
        )
    )

    (
        event_df
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(
            SCHEMA_DRIFT_EVENTS_TABLE
        )
    )

    print(
        f"Schema event registrado | "
        f"environment={ENVIRONMENT} | "
        f"batch={batch_id} | "
        f"decision="
        f"{schema_decision['decision']}"
    )

In [0]:
def auto_evolve_schema_contract(
    incoming_df,
    schema_check
):

    require_environment()

    unknown_columns = (
        schema_check[
            "unknown_columns"
        ]
    )

    if not unknown_columns:

        print(
            "No existen columnas nuevas "
            "para evolucionar"
        )

        return

    incoming_schema = {
        field.name:
            field.dataType.simpleString()

        for field
        in incoming_df.schema.fields
    }

    new_columns_data = [
        (
            column_name,
            incoming_schema[column_name],
            False
        )

        for column_name
        in unknown_columns
    ]

    new_columns_df = (
        spark.createDataFrame(
            new_columns_data,
            [
                "column_name",
                "data_type",
                "is_required"
            ]
        )
        .withColumn(
            "created_at",
            F.current_timestamp()
        )
        .withColumn(
            "updated_at",
            F.current_timestamp()
        )
    )

    contract_table = (
        DeltaTable.forName(
            spark,
            SCHEMA_CONTRACT_TABLE
        )
    )

    (
        contract_table
        .alias("target")
        .merge(
            new_columns_df.alias(
                "source"
            ),
            (
                "target.column_name "
                "= source.column_name"
            )
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(
        f"Schema Contract evolucionado | "
        f"environment={ENVIRONMENT} | "
        f"new_columns={unknown_columns}"
    )

In [0]:
def normalize_optional_columns(
    incoming_df
):

    require_environment()

    required_columns, optional_columns = (
        load_schema_contract()
    )

    incoming_columns = set(
        incoming_df.columns
    )

    normalized_df = incoming_df

    for (
        column_name,
        data_type
    ) in optional_columns.items():

        if (
            column_name
            not in incoming_columns
        ):

            normalized_df = (
                normalized_df
                .withColumn(
                    column_name,
                    F.lit(None).cast(
                        data_type
                    )
                )
            )

    return normalized_df

In [0]:
def reconcile_schema_evolution(
    incoming_df
):

    require_environment()

    bronze_table_name = (
        BRONZE_TRIPS_TABLE
    )

    contract_table_name = (
        SCHEMA_CONTRACT_TABLE
    )

    incoming_schema = {
        field.name:
            field.dataType.simpleString()

        for field
        in incoming_df.schema.fields
    }

    for (
        column_name,
        incoming_type
    ) in incoming_schema.items():

        contract_rows = (
            spark.table(
                contract_table_name
            )
            .filter(
                F.col("column_name")
                == column_name
            )
            .select(
                "data_type",
                "is_required"
            )
            .collect()
        )

        if not contract_rows:

            raise ValueError(
                f"SCHEMA_RECONCILIATION_ERROR | "
                f"environment={ENVIRONMENT} | "
                f"column={column_name} | "
                f"column_not_found_in_contract"
            )

        contract_type = (
            contract_rows[0][
                "data_type"
            ]
        )

        if (
            contract_type
            != incoming_type
        ):

            raise ValueError(
                f"SCHEMA_RECONCILIATION_ERROR | "
                f"environment={ENVIRONMENT} | "
                f"column={column_name} | "
                f"contract_type={contract_type} | "
                f"incoming_type={incoming_type}"
            )

        bronze_schema = {
            field.name:
                field.dataType.simpleString()

            for field
            in spark.table(
                bronze_table_name
            ).schema.fields
        }

        if (
            column_name
            not in bronze_schema
        ):

            spark.sql(f"""
                ALTER TABLE
                    {bronze_table_name}

                ADD COLUMNS (
                    `{column_name}`
                    {incoming_type}
                )
            """)

            print(
                f"Bronze reconciliado | "
                f"environment={ENVIRONMENT} | "
                f"column={column_name} | "
                f"type={incoming_type}"
            )

        else:

            if (
                bronze_schema[column_name]
                != incoming_type
            ):

                raise ValueError(
                    f"SCHEMA_RECONCILIATION_ERROR | "
                    f"environment={ENVIRONMENT} | "
                    f"column={column_name} | "
                    f"bronze_type="
                    f"{bronze_schema[column_name]} | "
                    f"incoming_type="
                    f"{incoming_type}"
                )

        final_bronze_schema = {
            field.name:
                field.dataType.simpleString()

            for field
            in spark.table(
                bronze_table_name
            ).schema.fields
        }

        if (
            column_name
            not in final_bronze_schema
            or
            final_bronze_schema[column_name]
            != incoming_type
        ):

            raise ValueError(
                f"SCHEMA_RECONCILIATION_ERROR | "
                f"environment={ENVIRONMENT} | "
                f"column={column_name} | "
                f"bronze_contract_not_aligned"
            )

    print(
        f"Schema Contract y Bronze "
        f"reconciliados correctamente | "
        f"environment={ENVIRONMENT}"
    )

In [0]:
def validate_schema_gate_v2(
    incoming_df,
    batch_id
):

    require_environment()

    schema_check = (
        detect_schema_drift_v2(
            incoming_df
        )
    )

    schema_decision = (
        classify_schema_change_v2(
            schema_check
        )
    )

    decision = (
        schema_decision[
            "decision"
        ]
    )

    print(
        f"Schema decision | "
        f"environment={ENVIRONMENT} | "
        f"batch={batch_id} | "
        f"decision={decision}"
    )

    write_schema_drift_event_v2(
        batch_id=batch_id,
        schema_check=schema_check,
        schema_decision=schema_decision
    )

    if decision == "REJECT":

        raise ValueError(
            f"SCHEMA_DRIFT_ERROR | "
            f"environment={ENVIRONMENT} | "
            f"batch_id={batch_id} | "
            f"details={schema_check}"
        )

    if decision == "AUTO_EVOLVE":

        auto_evolve_schema_contract(
            incoming_df=incoming_df,
            schema_check=schema_check
        )

    reconcile_schema_evolution(
        incoming_df=incoming_df
    )

    normalized_df = (
        normalize_optional_columns(
            incoming_df
        )
    )

    return normalized_df

In [0]:
# =========================================================
# BRONZE
# =========================================================

def upsert_batch_control(
    batch_id,
    delivery_date,
    delivery_batch,
    landing_path,
    records_landing,
    records_bronze
):

    require_environment()

    spark.sql(f"""
        MERGE INTO {BATCH_CONTROL_TABLE} AS target

        USING (
            SELECT
                '{batch_id}' AS batch_id,
                DATE '{delivery_date}' AS delivery_date,
                '{delivery_batch}' AS delivery_batch,
                '{landing_path}' AS source_path,
                'SUCCESS' AS status,
                {records_landing} AS records_landing,
                {records_bronze} AS records_bronze,
                current_timestamp() AS processed_at
        ) AS source

        ON target.batch_id = source.batch_id

        WHEN MATCHED THEN
            UPDATE SET
                target.delivery_date = source.delivery_date,
                target.delivery_batch = source.delivery_batch,
                target.source_path = source.source_path,
                target.status = source.status,
                target.records_landing = source.records_landing,
                target.records_bronze = source.records_bronze,
                target.processed_at = source.processed_at

        WHEN NOT MATCHED THEN
            INSERT (
                batch_id,
                delivery_date,
                delivery_batch,
                source_path,
                status,
                records_landing,
                records_bronze,
                processed_at
            )
            VALUES (
                source.batch_id,
                source.delivery_date,
                source.delivery_batch,
                source.source_path,
                source.status,
                source.records_landing,
                source.records_bronze,
                source.processed_at
            )
    """)

In [0]:
def process_batch(
    delivery_date,
    delivery_batch,
    force_reprocess=False,
    run_type="INCREMENTAL"
):

    require_environment()

    # ---------------------------------------------------------
    # 1. Validar run_type
    # ---------------------------------------------------------

    allowed_run_types = [
        "INCREMENTAL",
        "BACKFILL",
        "REPROCESS"
    ]

    run_type = (
        str(run_type)
        .strip()
        .upper()
    )

    if run_type not in allowed_run_types:

        raise ValueError(
            f"INVALID_RUN_TYPE | "
            f"run_type={run_type} | "
            f"allowed={allowed_run_types}"
        )

    # ---------------------------------------------------------
    # 2. Construir identificadores del batch
    # ---------------------------------------------------------

    batch_id = (
        f"{delivery_date.replace('-', '')}_"
        f"{delivery_batch}"
    )

    run_id = str(uuid.uuid4())

    landing_path = (
        "gs://de-zoomcamp-501217-databricks/"
        "landing/nyc_taxi/yellow/"
        f"{delivery_date}/batch_{delivery_batch}/"
    )

    print(
        f"Procesando batch: {batch_id} | "
        f"environment={ENVIRONMENT} | "
        f"run_type={run_type}"
    )

    print(
        f"Run ID: {run_id}"
    )

    # ---------------------------------------------------------
    # 3. Consultar estado previo
    # ---------------------------------------------------------

    already_processed = (
        spark.table(
            BATCH_CONTROL_TABLE
        )
        .filter(
            (F.col("batch_id") == batch_id)
            &
            (F.col("status") == "SUCCESS")
        )
        .limit(1)
        .count()
        > 0
    )

    # ---------------------------------------------------------
    # 4. Reglas según tipo de ejecución
    # ---------------------------------------------------------

    if run_type == "INCREMENTAL":

        if already_processed:

            print(
                f"SKIP: {batch_id} "
                f"ya fue procesado correctamente "
                f"en environment={ENVIRONMENT}."
            )

            return

    elif run_type == "BACKFILL":

        if already_processed:

            raise ValueError(
                f"BACKFILL_ALREADY_PROCESSED | "
                f"environment={ENVIRONMENT} | "
                f"batch_id={batch_id} | "
                f"El batch ya tiene estado SUCCESS. "
                f"Usa REPROCESS si deseas reconstruirlo."
            )

        print(
            f"BACKFILL: procesando batch histórico "
            f"{batch_id} | "
            f"environment={ENVIRONMENT}"
        )

    elif run_type == "REPROCESS":

        if not already_processed:

            raise ValueError(
                f"REPROCESS_REQUIRES_SUCCESS | "
                f"environment={ENVIRONMENT} | "
                f"batch_id={batch_id} | "
                f"El batch no tiene un procesamiento "
                f"SUCCESS previo."
            )

        print(
            f"REPROCESS: reconstruyendo batch "
            f"{batch_id} | "
            f"environment={ENVIRONMENT}"
        )

        force_reprocess = True

    # ---------------------------------------------------------
    # 5. Registrar intento
    # ---------------------------------------------------------

    spark.sql(f"""
        INSERT INTO {PIPELINE_RUNS_TABLE}
        (
            run_id,
            batch_id,
            delivery_date,
            delivery_batch,
            status,
            started_at,
            finished_at,
            records_landing,
            records_bronze,
            error_message,
            run_type
        )

        VALUES (
            '{run_id}',
            '{batch_id}',
            DATE '{delivery_date}',
            '{delivery_batch}',
            'RUNNING',
            current_timestamp(),
            NULL,
            NULL,
            NULL,
            NULL,
            '{run_type}'
        )
    """)

    try:

        # -----------------------------------------------------
        # 6. Validar Landing
        # -----------------------------------------------------

        landing_state = (
            get_landing_status(
                delivery_date=delivery_date,
                delivery_batch=delivery_batch
            )
        )

        if (
            landing_state["status"]
            != "AVAILABLE"
        ):

            raise ValueError(
                f"LANDING_NOT_AVAILABLE | "
                f"environment={ENVIRONMENT} | "
                f"batch_id={batch_id} | "
                f"landing_status="
                f"{landing_state['status']} | "
                f"records="
                f"{landing_state['records']}"
            )

        # -----------------------------------------------------
        # 7. Leer Landing
        # -----------------------------------------------------

        df_landing = (
            spark.read.parquet(
                landing_path
            )
        )

        # -----------------------------------------------------
        # 8. Schema Gate
        # -----------------------------------------------------

        df_landing_ready = (
            validate_schema_gate_v2(
                incoming_df=df_landing,
                batch_id=batch_id
            )
        )

        # -----------------------------------------------------
        # 9. Determinar corte esperado
        # -----------------------------------------------------

        dropoff_ts = (
            F.col(
                "tpep_dropoff_datetime"
            ).cast("timestamp")
        )

        dropoff_hour = (
            F.hour(
                dropoff_ts
            )
        )

        expected_delivery_date = (
            F.when(
                dropoff_hour == 23,
                F.date_add(
                    F.to_date(dropoff_ts),
                    1
                )
            )
            .otherwise(
                F.to_date(dropoff_ts)
            )
        )

        expected_delivery_batch = (
            F.when(
                dropoff_hour < 8,
                F.lit("0800")
            )
            .when(
                dropoff_hour < 16,
                F.lit("1600")
            )
            .when(
                dropoff_hour < 23,
                F.lit("2300")
            )
            .otherwise(
                F.lit("0800")
            )
        )

        # -----------------------------------------------------
        # 10. Construir Bronze
        # -----------------------------------------------------

        df_bronze = (
            df_landing_ready

            .withColumn(
                "expected_delivery_date",
                expected_delivery_date
            )

            .withColumn(
                "expected_delivery_batch",
                expected_delivery_batch
            )

            .withColumn(
                "expected_batch_id",
                F.concat(
                    F.date_format(
                        F.col(
                            "expected_delivery_date"
                        ),
                        "yyyyMMdd"
                    ),
                    F.lit("_"),
                    F.col(
                        "expected_delivery_batch"
                    )
                )
            )

            .withColumn(
                "arrival_status",
                F.when(
                    F.col("expected_batch_id")
                    ==
                    F.lit(batch_id),
                    F.lit("ON_TIME")
                )
                .otherwise(
                    F.lit("LATE")
                )
            )

            .withColumn(
                "ingestion_timestamp",
                F.current_timestamp()
            )

            .withColumn(
                "source_file",
                F.col(
                    "_metadata.file_path"
                )
            )
        )

        # -----------------------------------------------------
        # 11. Métricas late-arriving
        # -----------------------------------------------------

        arrival_metrics = (
            df_bronze
            .groupBy(
                "arrival_status"
            )
            .count()
            .collect()
        )

        arrival_counts = {
            row["arrival_status"]: row["count"]
            for row in arrival_metrics
        }

        on_time_records = (
            arrival_counts.get(
                "ON_TIME",
                0
            )
        )

        late_records = (
            arrival_counts.get(
                "LATE",
                0
            )
        )

        print(
            f"Arrival metrics | "
            f"environment={ENVIRONMENT} | "
            f"batch={batch_id} | "
            f"ON_TIME={on_time_records} | "
            f"LATE={late_records}"
        )

        # -----------------------------------------------------
        # 12. Contar Landing
        # -----------------------------------------------------

        records_landing = (
            df_landing.count()
        )

        # -----------------------------------------------------
        # 13. Escribir Bronze
        # -----------------------------------------------------

        (
            df_bronze
            .write
            .format("delta")
            .mode("overwrite")
            .option(
                "replaceWhere",
                f"batch_id = '{batch_id}'"
            )
            .option(
                "mergeSchema",
                "true"
            )
            .saveAsTable(
                BRONZE_TRIPS_TABLE
            )
        )

        # -----------------------------------------------------
        # 14. Contar Bronze
        # -----------------------------------------------------

        records_bronze = (
            spark.table(
                BRONZE_TRIPS_TABLE
            )
            .filter(
                F.col("batch_id")
                == batch_id
            )
            .count()
        )

        # -----------------------------------------------------
        # 15. Completeness Landing -> Bronze
        # -----------------------------------------------------

        if (
            records_landing
            != records_bronze
        ):

            raise ValueError(
                f"COMPLETENESS_ERROR | "
                f"environment={ENVIRONMENT} | "
                f"batch_id={batch_id} | "
                f"Landing={records_landing} | "
                f"Bronze={records_bronze}"
            )

        # -----------------------------------------------------
        # 16. Batch Control
        # -----------------------------------------------------

        upsert_batch_control(
            batch_id=batch_id,
            delivery_date=delivery_date,
            delivery_batch=delivery_batch,
            landing_path=landing_path,
            records_landing=records_landing,
            records_bronze=records_bronze
        )

        # -----------------------------------------------------
        # 17. Finalizar pipeline run
        # -----------------------------------------------------

        spark.sql(f"""
            UPDATE {PIPELINE_RUNS_TABLE}

            SET
                status = 'SUCCESS',
                finished_at =
                    current_timestamp(),
                records_landing =
                    {records_landing},
                records_bronze =
                    {records_bronze}

            WHERE
                run_id = '{run_id}'
        """)

        print(
            f"SUCCESS: {batch_id} | "
            f"environment={ENVIRONMENT} | "
            f"run_type={run_type} | "
            f"Landing={records_landing} | "
            f"Bronze={records_bronze} | "
            f"ON_TIME={on_time_records} | "
            f"LATE={late_records}"
        )

    except Exception as e:

        error_message = (
            str(e)
            .replace("'", "''")
        )

        spark.sql(f"""
            UPDATE {PIPELINE_RUNS_TABLE}

            SET
                status = 'FAILED',
                finished_at =
                    current_timestamp(),
                error_message =
                    '{error_message}'

            WHERE
                run_id = '{run_id}'
        """)

        print(
            f"FAILED: {batch_id} | "
            f"environment={ENVIRONMENT} | "
            f"run_type={run_type}"
        )

        print(
            error_message
        )

        raise

In [0]:
def run_bronze_task(
    delivery_date,
    delivery_batch,
    force_reprocess=False,
    run_type="INCREMENTAL"
):

    require_environment()

    batch_id = (
        delivery_date.replace("-", "")
        + "_"
        + delivery_batch
    )

    stage = "BRONZE"

    task_run_id = start_task_run(
        batch_id=batch_id,
        stage=stage
    )

    try:

        process_batch(
            delivery_date=delivery_date,
            delivery_batch=delivery_batch,
            force_reprocess=force_reprocess,
            run_type=run_type
        )

        finish_task_run(
            task_run_id=task_run_id,
            status="SUCCESS"
        )

        print(
            f"BRONZE finalizado correctamente | "
            f"environment={ENVIRONMENT} | "
            f"batch={batch_id} | "
            f"run_type={run_type}"
        )

    except Exception as e:

        finish_task_run(
            task_run_id=task_run_id,
            status="FAILED",
            error_message=str(e)
        )

        print(
            f"ERROR en BRONZE | "
            f"environment={ENVIRONMENT} | "
            f"batch={batch_id} | "
            f"run_type={run_type}"
        )

        print(
            str(e)
        )

        raise

In [0]:
# =========================================================
# SILVER
# =========================================================

def transform_silver_batch(batch_id):

    require_environment()

    # ---------------------------------------------------------
    # 1. Leer únicamente el batch solicitado desde Bronze
    # ---------------------------------------------------------

    batch_df = (
        spark.table(
            BRONZE_TRIPS_TABLE
        )
        .filter(
            F.col("batch_id") == batch_id
        )
    )

    # ---------------------------------------------------------
    # 2. Normalización
    # ---------------------------------------------------------

    df = (
        batch_df

        .withColumn(
            "store_and_fwd_flag",
            F.upper(
                F.trim(
                    F.col("store_and_fwd_flag")
                )
            )
        )

        .withColumn(
            "tpep_pickup_datetime",
            F.col(
                "tpep_pickup_datetime"
            ).cast("timestamp")
        )

        .withColumn(
            "tpep_dropoff_datetime",
            F.col(
                "tpep_dropoff_datetime"
            ).cast("timestamp")
        )

        .withColumn(
            "VendorID",
            F.col("VendorID").cast("int")
        )

        .withColumn(
            "passenger_count",
            F.col(
                "passenger_count"
            ).cast("int")
        )

        .withColumn(
            "RatecodeID",
            F.col("RatecodeID").cast("int")
        )

        .withColumn(
            "PULocationID",
            F.col(
                "PULocationID"
            ).cast("int")
        )

        .withColumn(
            "DOLocationID",
            F.col(
                "DOLocationID"
            ).cast("int")
        )

        .withColumn(
            "payment_type",
            F.col(
                "payment_type"
            ).cast("int")
        )

        .withColumn(
            "trip_distance",
            F.col(
                "trip_distance"
            ).cast("decimal(12,3)")
        )

        .withColumn(
            "fare_amount",
            F.col(
                "fare_amount"
            ).cast("decimal(12,2)")
        )

        .withColumn(
            "extra",
            F.col("extra").cast(
                "decimal(12,2)"
            )
        )

        .withColumn(
            "mta_tax",
            F.col("mta_tax").cast(
                "decimal(12,2)"
            )
        )

        .withColumn(
            "tip_amount",
            F.col("tip_amount").cast(
                "decimal(12,2)"
            )
        )

        .withColumn(
            "tolls_amount",
            F.col("tolls_amount").cast(
                "decimal(12,2)"
            )
        )

        .withColumn(
            "improvement_surcharge",
            F.col(
                "improvement_surcharge"
            ).cast("decimal(12,2)")
        )

        .withColumn(
            "total_amount",
            F.col(
                "total_amount"
            ).cast("decimal(12,2)")
        )

        .withColumn(
            "congestion_surcharge",
            F.col(
                "congestion_surcharge"
            ).cast("decimal(12,2)")
        )

        .withColumn(
            "Airport_fee",
            F.col(
                "Airport_fee"
            ).cast("decimal(12,2)")
        )
    )

    # ---------------------------------------------------------
    # 3. trip_business_key
    # ---------------------------------------------------------

    df = df.withColumn(
        "trip_business_key",
        F.sha2(
            F.concat_ws(
                "||",

                F.coalesce(
                    F.col("VendorID").cast("string"),
                    F.lit("__NULL__")
                ),

                F.coalesce(
                    F.date_format(
                        F.col(
                            "tpep_pickup_datetime"
                        ),
                        "yyyy-MM-dd HH:mm:ss"
                    ),
                    F.lit("__NULL__")
                ),

                F.coalesce(
                    F.date_format(
                        F.col(
                            "tpep_dropoff_datetime"
                        ),
                        "yyyy-MM-dd HH:mm:ss"
                    ),
                    F.lit("__NULL__")
                ),

                F.coalesce(
                    F.col(
                        "PULocationID"
                    ).cast("string"),
                    F.lit("__NULL__")
                ),

                F.coalesce(
                    F.col(
                        "DOLocationID"
                    ).cast("string"),
                    F.lit("__NULL__")
                )
            ),
            256
        )
    )

    # ---------------------------------------------------------
    # 4. record_key
    # ---------------------------------------------------------

    df = df.withColumn(
        "record_key",
        F.sha2(
            F.concat_ws(
                "||",

                F.coalesce(F.col("VendorID").cast("string"), F.lit("__NULL__")),
                F.coalesce(F.date_format(F.col("tpep_pickup_datetime"), "yyyy-MM-dd HH:mm:ss"), F.lit("__NULL__")),
                F.coalesce(F.date_format(F.col("tpep_dropoff_datetime"), "yyyy-MM-dd HH:mm:ss"), F.lit("__NULL__")),
                F.coalesce(F.col("passenger_count").cast("string"), F.lit("__NULL__")),
                F.coalesce(F.col("trip_distance").cast("string"), F.lit("__NULL__")),
                F.coalesce(F.col("RatecodeID").cast("string"), F.lit("__NULL__")),
                F.coalesce(F.col("store_and_fwd_flag"), F.lit("__NULL__")),
                F.coalesce(F.col("PULocationID").cast("string"), F.lit("__NULL__")),
                F.coalesce(F.col("DOLocationID").cast("string"), F.lit("__NULL__")),
                F.coalesce(F.col("payment_type").cast("string"), F.lit("__NULL__")),
                F.coalesce(F.col("fare_amount").cast("string"), F.lit("__NULL__")),
                F.coalesce(F.col("extra").cast("string"), F.lit("__NULL__")),
                F.coalesce(F.col("mta_tax").cast("string"), F.lit("__NULL__")),
                F.coalesce(F.col("tip_amount").cast("string"), F.lit("__NULL__")),
                F.coalesce(F.col("tolls_amount").cast("string"), F.lit("__NULL__")),
                F.coalesce(F.col("improvement_surcharge").cast("string"), F.lit("__NULL__")),
                F.coalesce(F.col("total_amount").cast("string"), F.lit("__NULL__")),
                F.coalesce(F.col("congestion_surcharge").cast("string"), F.lit("__NULL__")),
                F.coalesce(F.col("Airport_fee").cast("string"), F.lit("__NULL__"))
            ),
            256
        )
    )

    # ---------------------------------------------------------
    # 5. Reglas de Data Quality
    # ---------------------------------------------------------

    df = (
        df

        .withColumn(
            "dq_vendor_not_null",
            F.col("VendorID").isNotNull()
        )

        .withColumn(
            "dq_pickup_not_null",
            F.col(
                "tpep_pickup_datetime"
            ).isNotNull()
        )

        .withColumn(
            "dq_dropoff_not_null",
            F.col(
                "tpep_dropoff_datetime"
            ).isNotNull()
        )

        .withColumn(
            "dq_pu_location_not_null",
            F.col(
                "PULocationID"
            ).isNotNull()
        )

        .withColumn(
            "dq_do_location_not_null",
            F.col(
                "DOLocationID"
            ).isNotNull()
        )

        .withColumn(
            "dq_valid_datetime",
            (
                F.col(
                    "tpep_dropoff_datetime"
                ).isNotNull()
                &
                F.col(
                    "tpep_pickup_datetime"
                ).isNotNull()
                &
                (
                    F.col(
                        "tpep_dropoff_datetime"
                    )
                    >=
                    F.col(
                        "tpep_pickup_datetime"
                    )
                )
            )
        )

        .withColumn(
            "dq_valid_distance",
            F.col("trip_distance").isNull()
            |
            (F.col("trip_distance") >= 0)
        )

        .withColumn(
            "dq_valid_passenger_count",
            F.col("passenger_count").isNull()
            |
            (F.col("passenger_count") >= 0)
        )
    )

    # ---------------------------------------------------------
    # 6. Bandera final DQ
    # ---------------------------------------------------------

    df = df.withColumn(
        "dq_is_valid",
        (
            F.col("dq_vendor_not_null")
            &
            F.col("dq_pickup_not_null")
            &
            F.col("dq_dropoff_not_null")
            &
            F.col("dq_pu_location_not_null")
            &
            F.col("dq_do_location_not_null")
            &
            F.col("dq_valid_datetime")
            &
            F.col("dq_valid_distance")
            &
            F.col(
                "dq_valid_passenger_count"
            )
        )
    )

    # ---------------------------------------------------------
    # 7. Todos los motivos de fallo
    # ---------------------------------------------------------

    df = df.withColumn(
        "dq_failure_reason",
        F.array_compact(
            F.array(

                F.when(
                    ~F.col("dq_vendor_not_null"),
                    F.lit("VENDOR_NULL")
                ),

                F.when(
                    ~F.col("dq_pickup_not_null"),
                    F.lit("PICKUP_DATETIME_NULL")
                ),

                F.when(
                    ~F.col("dq_dropoff_not_null"),
                    F.lit("DROPOFF_DATETIME_NULL")
                ),

                F.when(
                    ~F.col(
                        "dq_pu_location_not_null"
                    ),
                    F.lit("PU_LOCATION_NULL")
                ),

                F.when(
                    ~F.col(
                        "dq_do_location_not_null"
                    ),
                    F.lit("DO_LOCATION_NULL")
                ),

                F.when(
                    ~F.col("dq_valid_datetime"),
                    F.lit(
                        "INVALID_TRIP_DATETIME"
                    )
                ),

                F.when(
                    ~F.col("dq_valid_distance"),
                    F.lit(
                        "INVALID_TRIP_DISTANCE"
                    )
                ),

                F.when(
                    ~F.col(
                        "dq_valid_passenger_count"
                    ),
                    F.lit(
                        "INVALID_PASSENGER_COUNT"
                    )
                )
            )
        )
    )

    return df

In [0]:
def write_silver_batch(batch_df):

    require_environment()

    valid_df = (
        batch_df
        .filter(
            F.col("dq_is_valid") == True
        )
    )

    invalid_df = (
        batch_df
        .filter(
            F.col("dq_is_valid") == False
        )
    )

    silver_target = (
        DeltaTable.forName(
            spark,
            SILVER_TRIPS_TABLE
        )
    )

    (
        silver_target
        .alias("target")
        .merge(
            valid_df.alias("source"),
            (
                "target.record_key "
                "= source.record_key"
            )
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    quarantine_target = (
        DeltaTable.forName(
            spark,
            QUARANTINE_TABLE
        )
    )

    (
        quarantine_target
        .alias("target")
        .merge(
            invalid_df.alias("source"),
            (
                "target.record_key "
                "= source.record_key"
            )
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

In [0]:
def calculate_dq_metrics(batch_df):

    metrics_df = (
        batch_df
        .groupBy(
            "batch_id",
            "delivery_date",
            "delivery_batch"
        )
        .agg(
            F.count("*").alias("total_records"),
            F.sum(F.col("dq_is_valid").cast("int")).alias("valid_records"),
            F.sum((~F.col("dq_is_valid")).cast("int")).alias("invalid_records"),
            F.sum((~F.col("dq_vendor_not_null")).cast("int")).alias("fail_vendor_null"),
            F.sum((~F.col("dq_pickup_not_null")).cast("int")).alias("fail_pickup_null"),
            F.sum((~F.col("dq_dropoff_not_null")).cast("int")).alias("fail_dropoff_null"),
            F.sum((~F.col("dq_pu_location_not_null")).cast("int")).alias("fail_pu_null"),
            F.sum((~F.col("dq_do_location_not_null")).cast("int")).alias("fail_do_null"),
            F.sum((~F.col("dq_valid_datetime")).cast("int")).alias("fail_datetime"),
            F.sum((~F.col("dq_valid_distance")).cast("int")).alias("fail_distance"),
            F.sum((~F.col("dq_valid_passenger_count")).cast("int")).alias("fail_passenger_count")
        )
    )

    metrics_df = (
        metrics_df
        .withColumn(
            "error_rate",
            F.col("invalid_records") / F.col("total_records")
        )
        .withColumn(
            "measured_at",
            F.current_timestamp()
        )
    )

    return metrics_df

In [0]:
def write_dq_metrics(metrics_df):

    require_environment()

    dq_target = (
        DeltaTable.forName(
            spark,
            DQ_METRICS_TABLE
        )
    )

    (
        dq_target
        .alias("target")
        .merge(
            metrics_df.alias("source"),
            (
                "target.batch_id "
                "= source.batch_id"
            )
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

In [0]:
def clear_silver_batch(batch_id):

    require_environment()

    print(
        f"Limpiando resultados anteriores "
        f"de Silver | "
        f"environment={ENVIRONMENT} | "
        f"batch={batch_id}"
    )

    spark.sql(f"""
        DELETE FROM {SILVER_TRIPS_TABLE}
        WHERE batch_id = '{batch_id}'
    """)

    spark.sql(f"""
        DELETE FROM {QUARANTINE_TABLE}
        WHERE batch_id = '{batch_id}'
    """)

    print(
        f"Silver y Quarantine limpiados | "
        f"environment={ENVIRONMENT} | "
        f"batch={batch_id}"
    )

In [0]:
def process_silver_batch(
    batch_id,
    force_reprocess=False
):

    require_environment()

    stage = "SILVER"

    task_run_id = start_task_run(
        batch_id=batch_id,
        stage=stage
    )

    print(
        f"=== Procesando Silver | "
        f"environment={ENVIRONMENT} | "
        f"batch={batch_id} ==="
    )

    try:

        batch_df = (
            transform_silver_batch(
                batch_id
            )
        )

        total_records = (
            batch_df.count()
        )

        if total_records == 0:

            raise ValueError(
                f"BRONZE_EMPTY | "
                f"environment={ENVIRONMENT} | "
                f"batch_id={batch_id}"
            )

        if force_reprocess:

            print(
                f"REPROCESS Silver solicitado | "
                f"environment={ENVIRONMENT} | "
                f"batch={batch_id}"
            )

            clear_silver_batch(
                batch_id=batch_id
            )

        metrics_df = (
            calculate_dq_metrics(
                batch_df
            )
        )

        write_silver_batch(
            batch_df
        )

        write_dq_metrics(
            metrics_df
        )

        finish_task_run(
            task_run_id=task_run_id,
            status="SUCCESS"
        )

        print(
            f"Silver finalizado correctamente | "
            f"environment={ENVIRONMENT} | "
            f"batch={batch_id} | "
            f"total_evaluado={total_records}"
        )

    except Exception as e:

        finish_task_run(
            task_run_id=task_run_id,
            status="FAILED",
            error_message=str(e)
        )

        print(
            f"ERROR procesando Silver | "
            f"environment={ENVIRONMENT} | "
            f"batch={batch_id}"
        )

        print(
            str(e)
        )

        raise

In [0]:
def write_completeness_metrics(batch_id):

    require_environment()

    print(
        f"Calculando completeness | "
        f"environment={ENVIRONMENT} | "
        f"batch={batch_id}"
    )

    # ---------------------------------------------------------
    # 1. Métricas Landing -> Bronze
    # ---------------------------------------------------------

    batch_control_df = (
        spark.table(
            BATCH_CONTROL_TABLE
        )
        .filter(
            (F.col("batch_id") == batch_id)
            &
            (F.col("status") == "SUCCESS")
        )
        .select(
            "batch_id",
            "records_landing",
            "records_bronze"
        )
    )

    # ---------------------------------------------------------
    # 2. Métricas DQ del batch
    # ---------------------------------------------------------

    dq_df = (
        spark.table(
            DQ_METRICS_TABLE
        )
        .filter(
            F.col("batch_id") == batch_id
        )
        .select(
            "batch_id",
            "total_records",
            "valid_records",
            "invalid_records"
        )
    )

    # ---------------------------------------------------------
    # 3. Reconstruir record_key esperado desde Bronze
    #
    # Utilizamos exactamente la misma transformación que Silver.
    # ---------------------------------------------------------

    expected_df = (
        transform_silver_batch(
            batch_id=batch_id
        )
        .select(
            "record_key",
            "dq_is_valid"
        )
    )

    # ---------------------------------------------------------
    # 4. Record keys válidos esperados
    # ---------------------------------------------------------

    expected_valid_df = (
        expected_df
        .filter(
            F.col("dq_is_valid") == True
        )
        .select(
            "record_key"
        )
        .distinct()
    )

    # ---------------------------------------------------------
    # 5. Record keys inválidos esperados
    # ---------------------------------------------------------

    expected_invalid_df = (
        expected_df
        .filter(
            F.col("dq_is_valid") == False
        )
        .select(
            "record_key"
        )
        .distinct()
    )

    # ---------------------------------------------------------
    # 6. Record keys existentes en Silver
    # ---------------------------------------------------------

    silver_keys_df = (
        spark.table(
            SILVER_TRIPS_TABLE
        )
        .select(
            "record_key"
        )
        .distinct()
    )

    # ---------------------------------------------------------
    # 7. Record keys existentes en Quarantine
    # ---------------------------------------------------------

    quarantine_keys_df = (
        spark.table(
            QUARANTINE_TABLE
        )
        .select(
            "record_key"
        )
        .distinct()
    )

    # ---------------------------------------------------------
    # 8. Válidos esperados que NO existen en Silver
    # ---------------------------------------------------------

    missing_valid_df = (
        expected_valid_df
        .join(
            silver_keys_df,
            on="record_key",
            how="left_anti"
        )
    )

    # ---------------------------------------------------------
    # 9. Inválidos esperados que NO existen en Quarantine
    # ---------------------------------------------------------

    missing_invalid_df = (
        expected_invalid_df
        .join(
            quarantine_keys_df,
            on="record_key",
            how="left_anti"
        )
    )

    # ---------------------------------------------------------
    # 10. Contar faltantes físicos
    # ---------------------------------------------------------

    missing_valid_records = (
        missing_valid_df.count()
    )

    missing_invalid_records = (
        missing_invalid_df.count()
    )

    missing_materialized_records = (
        missing_valid_records
        +
        missing_invalid_records
    )

    # ---------------------------------------------------------
    # 11. Construir métricas de completeness
    # ---------------------------------------------------------

    completeness_df = (
        batch_control_df

        .join(
            dq_df,
            on="batch_id",
            how="inner"
        )

        .withColumn(
            "landing_bronze_ok",
            F.col("records_landing")
            ==
            F.col("records_bronze")
        )

        .withColumn(
            "bronze_silver_ok",
            (
                (
                    F.col("records_bronze")
                    ==
                    F.col("total_records")
                )
                &
                (
                    F.col("total_records")
                    ==
                    (
                        F.col("valid_records")
                        +
                        F.col("invalid_records")
                    )
                )
            )
        )

        .withColumn(
            "materialization_ok",
            F.lit(
                missing_materialized_records == 0
            )
        )

        .withColumn(
            "missing_valid_records",
            F.lit(
                missing_valid_records
            ).cast("long")
        )

        .withColumn(
            "missing_invalid_records",
            F.lit(
                missing_invalid_records
            ).cast("long")
        )

        .withColumn(
            "missing_materialized_records",
            F.lit(
                missing_materialized_records
            ).cast("long")
        )

        .withColumn(
            "completeness_status",
            F.when(
                F.col("landing_bronze_ok")
                &
                F.col("bronze_silver_ok")
                &
                F.col("materialization_ok"),
                F.lit("SUCCESS")
            )
            .otherwise(
                F.lit("FAILED")
            )
        )

        .withColumn(
            "measured_at",
            F.current_timestamp()
        )
    )

    # ---------------------------------------------------------
    # 12. Validar que existan inputs
    # ---------------------------------------------------------

    completeness_rows = (
        completeness_df
        .limit(1)
        .collect()
    )

    if not completeness_rows:

        raise ValueError(
            f"COMPLETENESS_INPUT_MISSING | "
            f"environment={ENVIRONMENT} | "
            f"batch_id={batch_id}"
        )

    completeness_row = (
        completeness_rows[0]
    )

    # ---------------------------------------------------------
    # 13. Persistir métricas
    # ---------------------------------------------------------

    completeness_target = (
        DeltaTable.forName(
            spark,
            COMPLETENESS_METRICS_TABLE
        )
    )

    (
        completeness_target
        .alias("target")
        .merge(
            completeness_df.alias("source"),
            (
                "target.batch_id "
                "= source.batch_id"
            )
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(
        f"Completeness registrado | "
        f"environment={ENVIRONMENT} | "
        f"batch={batch_id}"
    )

    completeness_df.show(
        truncate=False
    )

    # ---------------------------------------------------------
    # 14. Quality Gate
    # ---------------------------------------------------------

    if (
        completeness_row[
            "completeness_status"
        ]
        != "SUCCESS"
    ):

        raise ValueError(
            f"COMPLETENESS_FAILED | "
            f"environment={ENVIRONMENT} | "
            f"batch_id={batch_id} | "
            f"landing_bronze_ok="
            f"{completeness_row['landing_bronze_ok']} | "
            f"bronze_silver_ok="
            f"{completeness_row['bronze_silver_ok']} | "
            f"materialization_ok="
            f"{completeness_row['materialization_ok']} | "
            f"records_landing="
            f"{completeness_row['records_landing']} | "
            f"records_bronze="
            f"{completeness_row['records_bronze']} | "
            f"total_records="
            f"{completeness_row['total_records']} | "
            f"valid_records="
            f"{completeness_row['valid_records']} | "
            f"invalid_records="
            f"{completeness_row['invalid_records']} | "
            f"missing_valid_records="
            f"{completeness_row['missing_valid_records']} | "
            f"missing_invalid_records="
            f"{completeness_row['missing_invalid_records']} | "
            f"missing_materialized_records="
            f"{completeness_row['missing_materialized_records']}"
        )

    print(
        f"QUALITY GATE PASSED | "
        f"environment={ENVIRONMENT} | "
        f"batch={batch_id} | "
        f"missing_materialized_records=0"
    )

In [0]:
def run_completeness_task(batch_id):

    require_environment()

    stage = "COMPLETENESS"

    task_run_id = start_task_run(
        batch_id=batch_id,
        stage=stage
    )

    try:

        write_completeness_metrics(
            batch_id=batch_id
        )

        finish_task_run(
            task_run_id=task_run_id,
            status="SUCCESS"
        )

        print(
            f"COMPLETENESS finalizado correctamente | "
            f"environment={ENVIRONMENT} | "
            f"batch={batch_id}"
        )

    except Exception as e:

        finish_task_run(
            task_run_id=task_run_id,
            status="FAILED",
            error_message=str(e)
        )

        print(
            f"ERROR en COMPLETENESS | "
            f"environment={ENVIRONMENT} | "
            f"batch={batch_id}"
        )

        print(
            str(e)
        )

        raise

In [0]:
# =========================================================
# GOLD
# =========================================================
def process_gold_batch(batch_id):

    require_environment()

    print(
        f"Procesando Gold | "
        f"environment={ENVIRONMENT} | "
        f"batch={batch_id}"
    )

    # ---------------------------------------------------------
    # 1. Determinar fechas afectadas desde Bronze
    #
    # Bronze representa todos los registros recibidos por
    # el batch, incluso aquellos enviados a Quarantine.
    # ---------------------------------------------------------

    affected_dates_df = (
        spark.table(
            BRONZE_TRIPS_TABLE
        )
        .filter(
            F.col("batch_id") == batch_id
        )
        .filter(
            F.col(
                "tpep_dropoff_datetime"
            ).isNotNull()
        )
        .select(
            F.to_date(
                "tpep_dropoff_datetime"
            ).alias(
                "event_date"
            )
        )
        .filter(
            F.col(
                "event_date"
            ).isNotNull()
        )
        .distinct()
    )

    # ---------------------------------------------------------
    # 2. Materializar fechas afectadas
    # ---------------------------------------------------------

    affected_dates = (
        affected_dates_df
        .collect()
    )

    if not affected_dates:

        raise ValueError(
            f"GOLD_NO_AFFECTED_DATES | "
            f"environment={ENVIRONMENT} | "
            f"batch_id={batch_id}"
        )

    print(
        f"Fechas afectadas | "
        f"environment={ENVIRONMENT} | "
        f"batch={batch_id} | "
        f"count={len(affected_dates)}"
    )

    # ---------------------------------------------------------
    # 3. Leer estado completo de Silver
    # ---------------------------------------------------------

    silver_df = (
        spark.table(
            SILVER_TRIPS_TABLE
        )
        .withColumn(
            "event_date",
            F.to_date(
                "tpep_dropoff_datetime"
            )
        )
    )

    # ---------------------------------------------------------
    # 4. Filtrar Silver usando BROADCAST JOIN
    #
    # affected_dates_df es pequeño frente a Silver.
    # ---------------------------------------------------------

    affected_silver_df = (
        silver_df
        .join(
            F.broadcast(
                affected_dates_df
            ),
            on="event_date",
            how="left_semi"
        )
    )

    # ---------------------------------------------------------
    # 5. Recalcular Gold usando TODO Silver correspondiente
    #    a las fechas afectadas
    # ---------------------------------------------------------

    gold_updates_df = (
        affected_silver_df
        .groupBy(
            "event_date"
        )
        .agg(
            F.count("*").alias(
                "trip_count"
            ),

            F.round(
                F.sum("total_amount"),
                2
            ).alias(
                "total_revenue"
            ),

            F.round(
                F.sum("trip_distance"),
                2
            ).alias(
                "total_distance"
            ),

            F.round(
                F.avg("fare_amount"),
                2
            ).alias(
                "avg_fare_amount"
            ),

            F.round(
                F.avg("trip_distance"),
                2
            ).alias(
                "avg_trip_distance"
            )
        )
        .withColumn(
            "updated_at",
            F.current_timestamp()
        )
    )

    # ---------------------------------------------------------
    # 6. Fechas afectadas que todavía tienen registros válidos
    # ---------------------------------------------------------

    dates_with_valid_data_df = (
        gold_updates_df
        .select(
            "event_date"
        )
        .distinct()
    )

    # ---------------------------------------------------------
    # 7. Detectar fechas que quedaron sin registros válidos
    # ---------------------------------------------------------

    empty_affected_dates_df = (
        affected_dates_df
        .join(
            F.broadcast(
                dates_with_valid_data_df
            ),
            on="event_date",
            how="left_anti"
        )
    )

    empty_affected_dates = (
        empty_affected_dates_df
        .collect()
    )

    # ---------------------------------------------------------
    # 8. Obtener tabla Gold del environment activo
    # ---------------------------------------------------------

    gold_table = (
        DeltaTable.forName(
            spark,
            GOLD_DAILY_METRICS_TABLE
        )
    )

    # ---------------------------------------------------------
    # 9. Determinar si existen agregados para actualizar
    # ---------------------------------------------------------

    has_gold_updates = (
        gold_updates_df
        .limit(1)
        .count()
        > 0
    )

    # ---------------------------------------------------------
    # 10. MERGE de fechas con información válida
    # ---------------------------------------------------------

    if has_gold_updates:

        (
            gold_table
            .alias("target")
            .merge(
                gold_updates_df.alias(
                    "source"
                ),
                (
                    "target.event_date "
                    "= source.event_date"
                )
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

        print(
            f"Gold actualizado para fechas "
            f"con información válida | "
            f"environment={ENVIRONMENT}"
        )

    # ---------------------------------------------------------
    # 11. Eliminar métricas Gold obsoletas
    # ---------------------------------------------------------

    if empty_affected_dates:

        empty_dates = [
            row["event_date"]
            for row in empty_affected_dates
        ]

        empty_dates_sql = ", ".join(
            [
                f"DATE '{event_date}'"
                for event_date in empty_dates
            ]
        )

        spark.sql(f"""
            DELETE FROM {GOLD_DAILY_METRICS_TABLE}

            WHERE event_date IN (
                {empty_dates_sql}
            )
        """)

        print(
            f"Fechas eliminadas de Gold "
            f"por ausencia de registros válidos | "
            f"environment={ENVIRONMENT} | "
            f"dates={empty_dates}"
        )

    # ---------------------------------------------------------
    # 12. Resultado
    # ---------------------------------------------------------

    print(
        f"Gold actualizado correctamente | "
        f"environment={ENVIRONMENT} | "
        f"batch={batch_id}"
    )

    if has_gold_updates:

        gold_updates_df.orderBy(
            "event_date"
        ).show(
            truncate=False
        )

    else:

        print(
            f"El batch {batch_id} no dejó "
            f"registros válidos para Gold | "
            f"environment={ENVIRONMENT}"
        )


In [0]:
def run_gold_task(batch_id):

    require_environment()

    stage = "GOLD"

    task_run_id = start_task_run(
        batch_id=batch_id,
        stage=stage
    )

    try:

        process_gold_batch(
            batch_id=batch_id
        )

        finish_task_run(
            task_run_id=task_run_id,
            status="SUCCESS"
        )

        print(
            f"GOLD finalizado correctamente | "
            f"environment={ENVIRONMENT} | "
            f"batch={batch_id}"
        )

    except Exception as e:

        finish_task_run(
            task_run_id=task_run_id,
            status="FAILED",
            error_message=str(e)
        )

        print(
            f"ERROR en GOLD | "
            f"environment={ENVIRONMENT} | "
            f"batch={batch_id}"
        )

        print(
            str(e)
        )

        raise

In [0]:
def optimize_delta_tables(
    target_table=None
):

    require_environment()

    print(
        f"Iniciando mantenimiento Delta | "
        f"environment={ENVIRONMENT}"
    )

    # ---------------------------------------------------------
    # 1. Tablas autorizadas del environment activo
    # ---------------------------------------------------------

    allowed_tables = [
        BRONZE_TRIPS_TABLE,
        SILVER_TRIPS_TABLE,
        QUARANTINE_TABLE
    ]

    # ---------------------------------------------------------
    # 2. Determinar qué tablas procesar
    # ---------------------------------------------------------

    if target_table is None:

        tables_to_optimize = (
            allowed_tables
        )

        print(
            f"Modo mantenimiento: ALL_TABLES | "
            f"environment={ENVIRONMENT}"
        )

    else:

        target_table = (
            str(target_table)
            .strip()
        )

        if (
            target_table
            not in allowed_tables
        ):

            raise ValueError(
                f"INVALID_OPTIMIZE_TARGET | "
                f"environment={ENVIRONMENT} | "
                f"target_table={target_table} | "
                f"allowed_tables={allowed_tables}"
            )

        tables_to_optimize = [
            target_table
        ]

        print(
            f"Modo mantenimiento: "
            f"SINGLE_TABLE | "
            f"environment={ENVIRONMENT} | "
            f"target={target_table}"
        )

    # ---------------------------------------------------------
    # 3. Contenedores de resultados
    # ---------------------------------------------------------

    optimized_tables = []

    failed_tables = []

    # ---------------------------------------------------------
    # 4. Ejecutar OPTIMIZE
    # ---------------------------------------------------------

    for table_name in tables_to_optimize:

        print(
            f"OPTIMIZE START | "
            f"environment={ENVIRONMENT} | "
            f"table={table_name}"
        )

        try:

            spark.sql(
                f"""
                OPTIMIZE {table_name}
                """
            )

            optimized_tables.append(
                table_name
            )

            print(
                f"OPTIMIZE SUCCESS | "
                f"environment={ENVIRONMENT} | "
                f"table={table_name}"
            )

        except Exception as e:

            error_message = (
                str(e)
            )

            failed_tables.append(
                {
                    "table":
                        table_name,

                    "error":
                        error_message
                }
            )

            print(
                f"OPTIMIZE FAILED | "
                f"environment={ENVIRONMENT} | "
                f"table={table_name} | "
                f"error={error_message}"
            )

    # ---------------------------------------------------------
    # 5. Resumen
    # ---------------------------------------------------------

    print(
        f"Mantenimiento Delta finalizado | "
        f"environment={ENVIRONMENT}"
    )

    print(
        f"Tablas solicitadas: "
        f"{len(tables_to_optimize)}"
    )

    print(
        f"Tablas optimizadas: "
        f"{len(optimized_tables)}"
    )

    print(
        f"Tablas con error: "
        f"{len(failed_tables)}"
    )

    # ---------------------------------------------------------
    # 6. Propagar fallo
    # ---------------------------------------------------------

    if failed_tables:

        failed_table_names = [
            item["table"]
            for item in failed_tables
        ]

        raise RuntimeError(
            f"DELTA_OPTIMIZE_FAILED | "
            f"environment={ENVIRONMENT} | "
            f"failed_tables="
            f"{failed_table_names} | "
            f"details={failed_tables}"
        )

    # ---------------------------------------------------------
    # 7. Retornar resultado
    # ---------------------------------------------------------

    return {
        "status":
            "SUCCESS",

        "environment":
            ENVIRONMENT,

        "optimized_tables":
            optimized_tables,

        "failed_tables":
            []
    }